# Título do Projeto: [Nome claro e descritivo do projeto]

Objetivo: [Breve parágrafo a descrever o problema de negócio ou a pergunta de pesquisa que este projeto visa responder, como por exemplo, identificar fatores de risco para uma determinada condição.]

## bibliotecas

In [22]:
import pandas as pd
import os
import ftplib
from typing import List

## data bases


In [23]:
def download_datasus(
    estados: List[str] = None,
    anos: List[int] = None,
    sistema: str = "SIM",
    download_path: str = None
):
    raw_dir = download_path
    ftp_host = "ftp.datasus.gov.br"

    caminhos = {
        "SIM": "/dissemin/publicos/SIM/CID10/DORES/",
        "SIH": "/dissemin/publicos/SIHSUS/200801_/Dados/",
        "CNES": "/dissemin/publicos/CNES/200508_/Dados/"
    }

    if sistema not in caminhos:
        raise ValueError(f"Sistema {sistema} não suportado")

    ftp_path = caminhos[sistema]

    print(f"Conectando ao FTP: {ftp_host}")

    ftp = ftplib.FTP(ftp_host)
    ftp.login()
    ftp.cwd(ftp_path)

    if sistema == "CNES":
        anos_str = [str(ano)[-2:] for ano in anos]
        pastas = ftp.nlst()
        
        for pasta in pastas:
            caminho_pasta = f"{ftp_path}{pasta}/"
            ftp.cwd(caminho_pasta)
            arquivos = ftp.nlst()

            for arquivo in arquivos:
                if not arquivo.endswith(".dbc"):
                    continue

                valido_estado = False
                for estado in estados:
                    if estado in arquivo:
                        valido_estado = True
                        break
                
                if not valido_estado:
                    continue

                valido_ano = False
                for ano in anos_str:
                    if ano in arquivo:
                        valido_ano = True
                        break
                
                if not valido_ano:
                    continue

                local_path = os.path.join(raw_dir, arquivo)

                if os.path.exists(local_path):
                    continue

                try:
                    with open(local_path, "wb") as f:
                        ftp.retrbinary(f"RETR {arquivo}", f.write)
                except Exception:
                    pass
            
            ftp.cwd(ftp_path)

    else:
        meses = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
        for estado in estados:
            for ano in anos:
                ano_curto = str(ano)[-2:]
                for mes in meses:
                    if sistema == "SIM":
                        filename = f"DO{estado}{ano}.dbc"
                    elif sistema == "SIH":
                        filename = f"RD{estado}{ano_curto}{mes}.dbc"
                    
                    local_path = os.path.join(raw_dir, filename)

                    if os.path.exists(local_path):
                        if os.path.getsize(local_path) > 0:
                            continue
                        else:
                            os.remove(local_path)

                    print(f"Baixando: {filename}")

                    try:
                        with open(local_path, "wb") as f:
                            ftp.retrbinary(f"RETR {filename}", f.write)
                        print(f"Sucesso: {filename}")
                    except ftplib.error_perm:
                        if os.path.exists(local_path):
                            os.remove(local_path)
                        print(f"Arquivo não encontrado: {filename}")

    ftp.quit()
    print(f"\nDownload finalizado!")
    print(f"Arquivos salvos em: {raw_dir}")

In [24]:
# download_datasus(
#     estados=["SP"],
#     anos=[2016, 2026],
#     sistema="SIM",
#     download_path="C:\dsm\tcc\data\raw"
# )

In [25]:
download_datasus(
    estados=["SP"],
    anos=[2025],
    sistema="SIH",
    download_path= r"C:\dsm\tcc\data\raw\SIH"
)

Conectando ao FTP: ftp.datasus.gov.br
Baixando: RDSP2501.dbc
Sucesso: RDSP2501.dbc
Baixando: RDSP2502.dbc
Sucesso: RDSP2502.dbc
Baixando: RDSP2503.dbc
Sucesso: RDSP2503.dbc
Baixando: RDSP2504.dbc
Sucesso: RDSP2504.dbc
Baixando: RDSP2505.dbc
Sucesso: RDSP2505.dbc
Baixando: RDSP2506.dbc
Sucesso: RDSP2506.dbc
Baixando: RDSP2507.dbc
Sucesso: RDSP2507.dbc
Baixando: RDSP2508.dbc
Sucesso: RDSP2508.dbc
Baixando: RDSP2509.dbc
Sucesso: RDSP2509.dbc
Baixando: RDSP2510.dbc
Sucesso: RDSP2510.dbc
Baixando: RDSP2511.dbc
Sucesso: RDSP2511.dbc
Baixando: RDSP2512.dbc
Sucesso: RDSP2512.dbc

Download finalizado!
Arquivos salvos em: C:\dsm\tcc\data\raw\SIH


In [27]:
download_datasus(
    estados=["SP"],
    anos=[2025],
    sistema="CNES",
    download_path = r"C:\dsm\tcc\data\raw\CNES"
)

Conectando ao FTP: ftp.datasus.gov.br

Download finalizado!
Arquivos salvos em: C:\dsm\tcc\data\raw\CNES
